# Real-time online VJF: learning a limit cycle from a streaming Poisson population

This notebook learns a 2-D rotating limit cycle **online, one sample at a time**, from a
log-linear Poisson spike stream whose observation model `(C, b)` is **unknown** and recovered
on the fly. It demonstrates:

- a dependency-free synthetic spike stream (`vjf.synthetic`);
- the **projection encoder** + decoupled **online readout** (`OnlineReadout`): the recognition
  network reads `pinv(C) (g~(y) - b)` (an `m`-dim subspace coordinate) instead of the raw
  `n`-dim spike vector, with `C` estimated online by incremental PCA;
- the reusable per-sample loop `vjf.realtime.online_filter`, which threads the posterior,
  switches from warm-up to dynamics learning, guards against divergence, and times each step.

The script form is `examples/realtime_tutorial.py`.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from vjf import synthetic as syn
from vjf.model import VJF
from vjf.readout import OnlineReadout
from vjf.realtime import online_filter

SEED = 20260605
torch.manual_seed(SEED)

def affine_r2(mu, z):
    """R^2 of the best affine image of mu onto z (the VJF latent is affine-identifiable)."""
    X = np.concatenate([mu, np.ones((len(mu), 1))], axis=1)
    W, *_ = np.linalg.lstsq(X, z, rcond=None)
    sse = float(((z - X @ W) ** 2).sum()); tss = float(((z - z.mean(0)) ** 2).sum())
    return 1.0 - sse / (tss + 1e-12)

## 1. A synthetic spike stream

`limit_cycle` makes the 2-D latent `z`; `poisson_readout` sets a sparse log-linear loading
calibrated to biological rates (mean ~20 Hz, peak ~100 Hz at 5 ms bins); `stream` yields one
Poisson count vector per bin. The model never sees `C` or `b`.

In [ ]:
T, N = 4000, 60
z = syn.limit_cycle(T, dt=5e-3, angular_velocity=30.0, seed=SEED)
C, b = syn.poisson_readout(z, N, mean_rate=0.1, peak_rate=0.5, seed=SEED)
counts = np.array(list(syn.stream(z, C, b, seed=SEED + 1)))
rate = syn.rate_at(z, C, b)
print(f'mean rate/bin = {rate.mean():.3f}, peak rate/bin = {rate.max():.3f}, counts shape = {counts.shape}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(z[:600, 0], z[:600, 1], lw=0.8); ax[0].set_aspect('equal'); ax[0].set_title('latent z (3 s)')
# Diagnostic only: sort neurons by their TRUE preferred phase (uses oracle C; the model never sees this).
order = np.argsort(np.arctan2(C[:, 1], C[:, 0]))
ni, bi = np.nonzero(counts[:600, order].T)
ax[1].scatter(bi * 5e-3, ni, s=4, c='k', marker='|'); ax[1].set_title('spike raster (3 s)')
ax[1].set_xlabel('time (s)'); ax[1].set_ylabel('neuron (by true preferred phase)')
plt.tight_layout(); plt.show()

## 2. Model + a cheap readout warm-start

We build a VJF with the **projection encoder** and square-root-RLS dynamics. The first
`init_w` bins warm-start the `OnlineReadout` (a small batch PCA of the smoothed log-rate),
whose `(C, b)` we write into the frozen decoder. The readout then keeps refining `C` online.

In [ ]:
warmup_steps = 1000
model = VJF.make_model(N, 2, 0, n_rbf=100, hidden_sizes=[100, 100],
                       likelihood='poisson', transition_flow='srrls', encoder='projection')
ro = OnlineReadout(N, 2, smooth_tau=8.0, refresh_K=400, link='log')
init_w = min(10 * N, T // 2)                # cheap warm-start window
assert T > init_w + warmup_steps, 'need T > init_w + warmup_steps (shrink warmup or grow T)'
Cp, bp = ro.warm_start(counts[:init_w])
with torch.no_grad():
    model.decoder.decode.weight.copy_(torch.as_tensor(Cp))
    model.decoder.decode.bias.copy_(torch.as_tensor(bp.reshape(-1)))
model.decoder.requires_grad_(False)        # the online readout owns (C, b), not SGD
print(f'warm-start window = {init_w} bins; going live on the remaining {T - init_w} bins')

## 3. Go live: per-sample `online_filter`

We stream the **unseen remainder** (`counts[init_w:]`) through `online_filter`, one sample at
a time. Dynamics are frozen for `warmup_steps` while the recognition settles, then initialized
once and learned. We track a trailing-window affine R^2 and the steady per-bin compute time.

In [ ]:
align_window, log_every = 1500, 200
live = counts[init_w:]
z_live = z[init_w:init_w + len(live)]
means = np.zeros((len(live), 2), dtype=np.float32)
r2_steps, r2_vals, per_bin_ms = [], [], []
n_refresh = n_diverge = 0
for r in online_filter(model, live, readout=ro, warmup_steps=warmup_steps, rbf_width_scale=0.5):
    means[r.step] = r.mean
    n_refresh += int(r.refreshed); n_diverge += int(r.diverged)
    if not r.warming_up and not r.diverged:
        per_bin_ms.append(r.elapsed_s * 1e3)
    if (r.step + 1) % log_every == 0 and r.step + 1 > warmup_steps:
        lo = max(0, r.step + 1 - align_window)
        r2_steps.append(init_w + r.step + 1)
        r2_vals.append(affine_r2(means[lo:r.step + 1], z_live[lo:r.step + 1]))

mean_ms, p95_ms = float(np.mean(per_bin_ms)), float(np.percentile(per_bin_ms, 95))
budget = 'within' if p95_ms < 5.0 else 'over'
print(f'final trailing R^2 = {r2_vals[-1]:.3f}')
print(f'steady per-bin cost = {mean_ms:.3f} ms (p95 {p95_ms:.3f}) -- {budget} the 5 ms bin budget')
print(f'online (C,b) refreshes = {n_refresh}, diverged steps = {n_diverge}')

## 4. Results

The online estimate climbs as the readout subspace and the dynamics are learned; the aligned
VJF latent traces the true limit cycle. The alignment below is a post-hoc diagnostic (the VJF
latent is identifiable only up to an affine transform).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(r2_steps, r2_vals, lw=2); ax[0].set_ylim(min(-0.1, min(r2_vals) - 0.05), 1.0)
ax[0].set_xlabel('stream position (bins)'); ax[0].set_ylabel('trailing affine R^2')
ax[0].set_title('online latent recovery (unknown readout)')

# Diagnostic alignment: fit an affine map to the true z post-hoc (uses ground truth for
# visualization only; the online filter never sees z).
seg = slice(len(z_live) - 1500, len(z_live))
X = np.concatenate([means[seg], np.ones((len(means[seg]), 1))], 1)
aligned = X @ np.linalg.lstsq(X, z_live[seg], rcond=None)[0]
ax[1].plot(z_live[seg, 0], z_live[seg, 1], lw=0.8, alpha=0.7, label='true z')
ax[1].plot(aligned[:, 0], aligned[:, 1], lw=0.8, alpha=0.7, label='VJF (aligned)')
ax[1].set_aspect('equal'); ax[1].legend(); ax[1].set_title('phase portrait (last 1500 bins)')
plt.tight_layout(); plt.show()

## Takeaways

- VJF runs **per-sample online** in a few ms/bin; in this run it met the 5 ms budget (see the
  printed p95), so closed-loop use at 5 ms bins is feasible -- hardware-dependent.
- The **projection encoder** (`pinv(C)(g~-b)`) shrinks the recognition input from `n` spikes to
  `m` dimensions, and the **online readout** recovers the latent even though the observation
  model is unknown. (How close the online readout gets to the *known-readout* ceiling across SNR
  is quantified in `paper/`; this notebook does not run that oracle baseline.)
- The raster sorting and phase-portrait alignment are **diagnostics that use ground truth** for
  visualization; the online filter itself never sees `z`, `C`, or `b`.
- See `examples/realtime_tutorial.py` for the script form and `paper/` for the technical report.